# Setup CBAM-ResNet18 di Google Colab

Aktifkan **Runtime > Change runtime type > GPU**, lalu jalankan sel secara berurutan.
Notebook menyiapkan environment proyek dan satu langkah forward/backward GPU.
Tuning dan pelatihan penuh berjalan setelah opsi pada sel terkait diaktifkan. Model dan loader berasal dari modul repositori.

Gunakan URL HTTPS repositori GitHub yang dapat diakses. Data dan environment diunduh
ke runtime Colab; tidak ada penghubungan Google Drive otomatis.

In [ ]:
import importlib.metadata
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
from urllib.parse import urlparse

def run_logged(command, *, cwd):
    """Tampilkan stdout/stderr proses anak langsung pada output notebook."""
    child_env = os.environ.copy()
    child_env['MPLBACKEND'] = 'Agg'
    with subprocess.Popen(command, cwd=cwd, stdout=subprocess.PIPE,
                          stderr=subprocess.STDOUT, text=True, bufsize=1, env=child_env) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            returncode = process.wait()
        except KeyboardInterrupt:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait()
            raise
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, command)


if not Path('/content').is_dir() or sys.platform != 'linux':
    raise RuntimeError('Notebook setup ini ditujukan untuk runtime Google Colab Linux.')
if not (3, 12) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(f'Gunakan runtime Python 3.12 atau 3.13; saat ini {sys.version}.')
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('GPU NVIDIA belum tersedia. Pilih runtime GPU, lalu ulangi.')
subprocess.run(['nvidia-smi', '--query-gpu=name,driver_version,memory.total',
                '--format=csv,noheader'], check=True)
pip_version = importlib.metadata.version('pip')
if tuple(map(int, pip_version.split('.')[:2])) < (22, 3):
    raise RuntimeError('Perbarui pip runtime ke versi >=22.3 sebelum menjalankan setup.')
print(f'Python runtime: {sys.version.split()[0]}; pip: {pip_version}')


In [ ]:
REPO_DIR = Path('/content/cbam-resnet18-imagenette-160')
if not REPO_DIR.exists():
    repo_url = input('URL HTTPS repositori GitHub: ').strip()
    parsed = urlparse(repo_url)
    if (parsed.scheme != 'https' or parsed.hostname != 'github.com'
            or parsed.username or parsed.password or parsed.query or parsed.fragment
            or len(parsed.path.strip('/').split('/')) != 2):
        raise ValueError('Masukkan URL repositori, misalnya https://github.com/pemilik/nama-repo.git, tanpa token.')
    subprocess.run(['git', 'clone', '--', repo_url, str(REPO_DIR)], check=True)
for relative in ('train.py', 'tune_lr.py', 'check_env.py', 'requirements/common.txt', 'requirements/pytorch-cu118.txt'):
    if not (REPO_DIR / relative).is_file():
        raise RuntimeError(f'Berkas proyek tidak ditemukan: {REPO_DIR / relative}')
os.chdir(REPO_DIR)
print(f'Root proyek: {REPO_DIR}')


In [ ]:
# Venv terpisah menjaga paket bawaan notebook Colab tetap terpisah dari proyek.
VENV_DIR = REPO_DIR / '.venv'
COLAB_PYTHON = VENV_DIR / 'bin' / 'python'
if not COLAB_PYTHON.is_file():
    subprocess.run([sys.executable, '-m', 'venv', '--without-pip', str(VENV_DIR)], check=True)
subprocess.run([sys.executable, '-m', 'pip', '--python', str(COLAB_PYTHON),
                'install', 'pip==25.3'], check=True)
for requirements in ('requirements/pytorch-cu118.txt', 'requirements/common.txt'):
    subprocess.run([str(COLAB_PYTHON), '-m', 'pip', 'install', '--no-cache-dir',
                    '-r', requirements], cwd=REPO_DIR, check=True)
subprocess.run([str(COLAB_PYTHON), '-m', 'pip', 'check'], check=True)


In [ ]:
# Data lengkap yang sudah ada dipakai kembali oleh pengunduh.
print('Menyiapkan dataset; tunggu hingga pengunduhan dan pemeriksaan selesai.', flush=True)
result = subprocess.run([str(COLAB_PYTHON), '-u', '-m', 'utils.download_data'],
                        cwd=REPO_DIR, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout, end='', flush=True)
result.check_returncode()


In [ ]:
REPORT_PATH = REPO_DIR / 'outputs' / 'environment-colab.json'
result = subprocess.run([str(COLAB_PYTHON), '-u', 'check_env.py', '--require-cuda',
                        '--output', str(REPORT_PATH)], cwd=REPO_DIR,
                        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
print(result.stdout, end='', flush=True)
result.check_returncode()
report = json.loads(REPORT_PATH.read_text(encoding='utf-8'))
print(json.dumps({key: report[key] for key in ('status', 'device', 'dataset', 'smoke_test')},
                 indent=2, ensure_ascii=False))


## Tuning learning rate

Setelah pemeriksaan berhasil, ubah `RUN_TUNING = True` pada sel berikut.
Tiga kandidat **0.01, 0.05, 0.1** masing-masing dilatih **30 epoch**, memakai
seed 0 dan pembagian data yang sama. Tuning hanya membaca folder `train/`;
947 gambar validasi internal berasal dari folder tersebut.

LR dipilih dari akurasi validasi internal terbaik, lalu loss lebih rendah,
lalu LR lebih kecil jika keduanya sama. Data evaluasi akhir belum digunakan.

Untuk uji singkat sebelum tuning penuh, buat sel terpisah:

```python
subprocess.run([str(COLAB_PYTHON), 'tune_lr.py', '--epochs', '1',
                '--device', 'cuda', '--run-name', 'smoke-tuning'], cwd=REPO_DIR, check=True)
```

Hasil satu epoch hanya memeriksa alur, belum menentukan LR eksperimen utama.


In [ ]:
RUN_TUNING = False  # Ubah menjadi True untuk memulai 3 x 30 epoch.
TUNING_RUN = 'lr-search-30epoch'  # Gunakan nama baru jika hasil ini sudah ada.
TUNING_DIR = REPO_DIR / 'outputs' / TUNING_RUN

if RUN_TUNING:
    run_logged([str(COLAB_PYTHON), '-u', 'tune_lr.py', '--device', 'cuda',
                '--run-name', TUNING_RUN], cwd=REPO_DIR)
    comparison = json.loads((TUNING_DIR / 'results.json').read_text(encoding='utf-8'))
    print(json.dumps(comparison, indent=2, ensure_ascii=False))
else:
    print('Setup selesai. Aktifkan RUN_TUNING saat siap menjalankan tuning penuh.')


## Pelatihan utama

Setelah seluruh kandidat tuning selesai, aktifkan `RUN_MAIN_TRAINING`.
Sel ini membaca LR terpilih dari hasil tuning **30 epoch**, lalu melatih
model dari awal selama **30 epoch per seed (0, 1, 2)**. Checkpoint terbaik
tiap seed dipilih dari validasi internal dan dievaluasi pada 3.925 gambar akhir.


In [ ]:
RUN_MAIN_TRAINING = False  # Aktifkan setelah tuning penuh selesai.
MAIN_RUN = 'cbam-full-30epoch'

if RUN_MAIN_TRAINING:
    comparison = json.loads((TUNING_DIR / 'results.json').read_text(encoding='utf-8'))
    selected = json.loads((TUNING_DIR / 'selected_lr.json').read_text(encoding='utf-8'))
    if comparison['status'] != 'completed' or selected['tuning_epochs'] != 30:
        raise RuntimeError('Selesaikan tuning 30 epoch sebelum pelatihan utama.')
    print(f"Learning rate dari validasi internal: {selected['learning_rate']}")
    run_logged([str(COLAB_PYTHON), '-u', 'train.py', '--device', 'cuda',
                '--lr', str(selected['learning_rate']), '--epochs', '30',
                '--batch-size', str(selected['batch_size']), '--seeds', '0', '1', '2',
                '--run-name', MAIN_RUN], cwd=REPO_DIR)
else:
    print('Aktifkan RUN_MAIN_TRAINING setelah memeriksa hasil tuning.')


## Simpan hasil

Unduh atau simpan folder `outputs/` sebelum runtime Colab berakhir, termasuk
laporan environment, folder tuning, dan pelatihan utama.

- Tuning: `results.csv`, `results.json`, dan `selected_lr.json` berisi perbandingan LR.
- Setiap kandidat memiliki konfigurasi, split, checkpoint, history, dan grafik.
- Pelatihan utama juga menghasilkan metrik evaluasi akhir, confusion matrix,
  dan prediksi setiap gambar.

Gunakan nama run baru untuk percobaan berikutnya. Keberhasilan Colab perlu
diperiksa langsung pada runtime; pemeriksaan struktur notebook saja belum cukup.
